# A space in the total, and a bill that moved

`TotalCharges` is text. On 11 rows the text is a single space, not a number and not an empty field. Those 11 rows are exactly the rows with tenure 0, and every one of them has `Churn = No`. The first such id in file order is `4472-LVYGI`.

On the other 7032 rows the total can be read in cents. Compare it with tenure times the monthly charge, also in cents. The two integers agree on 614 rows. They disagree on $7032 - 614 = 6418$ rows.

Customer `7590-VHVEG` is one agreement: $1 \times 29.85 = 29.85$.

Customer `9305-CDSKC` is one disagreement. Tenure is 8 and the monthly charge is 99.65:

$$
8 \times 99.65 = 797.20.
$$

The stored total is 820.50. The gap is $820.50 - 797.20 = 23.30$.


In [1]:
# Locate the telco table beside this notebook, or under the course root.
import csv
from pathlib import Path

def data_path():
    here = Path.cwd()
    name = "Telco-Customer-Churn.csv"
    for folder in [here, *here.parents]:
        nested = folder / "16-Customer Churn" / "data" / name
        direct = folder / "data" / name
        if nested.exists():
            return nested
        if direct.exists():
            return direct
    raise FileNotFoundError(name)

def read_customers():
    with data_path().open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))
# Blank totals, and tenure times the monthly charge, both in cents.
rows = read_customers()

def cents(text):
    return int(round(float(text) * 100))

blanks = [row for row in rows if row["TotalCharges"] == " "]
assert len(blanks) == 11
assert all(row["tenure"] == "0" and row["Churn"] == "No" for row in blanks)
assert blanks[0]["customerID"] == "4472-LVYGI"
assert sum(row["tenure"] == "0" for row in rows) == 11
known = [row for row in rows if row["TotalCharges"] != " "]
assert len(known) == 7032
matches = 0
for row in known:
    product = int(row["tenure"]) * cents(row["MonthlyCharges"])
    if cents(row["TotalCharges"]) == product:
        matches += 1
print(matches, len(known) - matches)
assert matches == 614
assert 7032 - matches == 6418
first = rows[0]
assert cents(first["MonthlyCharges"]) == 2985
assert cents(first["TotalCharges"]) == 2985
by_id = {row["customerID"]: row for row in rows}
sample = by_id["9305-CDSKC"]
product = 8 * cents(sample["MonthlyCharges"])
total = cents(sample["TotalCharges"])
print(product, total, total - product)
assert product == 79720 and total == 82050 and total - product == 2330


614 6418
79720 82050 2330


## Pitfall

Filling the eleven spaces with the number 0 writes a total that the file does not contain. Those customers have tenure 0, so a bill has not been accumulated. Their label is Stay on all 11 rows.

## Summary

Eleven totals are the one-character string of a space, on the eleven tenure-0 rows, all labeled Stay. Among the 7032 numeric totals, 614 equal tenure times the current monthly charge. For `9305-CDSKC` the product is 797.20 and the stored total is 820.50.
